# Wifaq 1448 — Daily

**Once a day, after 12:00 pm** (1:00 pm from 1 Nov, when the free Gemini quota resets):
Runtime → **Change runtime type → T4 GPU** → Save, then Runtime → **Run all**. Nothing to edit.

Claude's scheduled card runs decide what this notebook does and write it to `My Drive/Wifaq 1448/automation/daily_config.json`. In order:

1. **Re-transcribe** the lessons whose transcripts are broken or have text missing (Gemini 3.8 / 3.7 / 3.6 Flash, your two keys, 10-minute parts, each part checked). The old transcript is kept in `transcripts/_replaced/`.
2. **Repair timestamps** of every transcript that hasn't been repaired yet (Whisper on the GPU, no Gemini quota). Hidaya 4 first.
3. **Opus copies** of the recordings for the card site.
4. Writes `automation/daily_done.md` for Claude.

When it says ✓ Finished, run your transcription notebook (v2.5) as usual: it uses whatever quota is left.

It is safe to stop and run again: finished work is skipped. If Colab says no GPU is available, try again later; step 1 still works on the CPU.

In [ ]:
# @title 1 · Connect Drive and install (2–3 minutes)
from google.colab import drive, userdata
drive.mount('/content/drive')
!pip -q install faster-whisper rapidfuzz google-genai
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none — Runtime → Change runtime type → T4 GPU (step 1 still works)")

In [ ]:
# @title 2 · Read today's to-do from Drive (written by Claude)
import os, json, datetime
DRIVE = "/content/drive/MyDrive"
OUT_DIR = f"{DRIVE}/Wifaq 1448"
TX_DIR = os.path.join(OUT_DIR, "transcripts")
AUTO_DIR = os.path.join(OUT_DIR, "automation")
os.makedirs(AUTO_DIR, exist_ok=True)

CONFIG = {"redo": [], "repair_books": ["ہدایہ جلد رابع"], "opus_books": ["ہدایہ جلد رابع"],
          "skip_books": ["ہدایہ جلد ثالث"], "whisper_model": "large-v3", "max_hours": 3}
try:
    with open(os.path.join(AUTO_DIR, "daily_config.json"), encoding="utf-8") as f:
        CONFIG.update(json.load(f))
    print("To-do updated by Claude:", CONFIG.get("updated", "?"))
except FileNotFoundError:
    print("No daily_config.json yet — using the defaults.")

# Settings used by the repair and Opus code below
BOOK_ORDER = ["ہدایہ جلد رابع", "مشکوۃ المصابیح جلد اول", "مشکوۃ المصابیح جلد دوم", "تیسیر مصطلح الحدیث",
              "شرح نخبۃ الفکر", "التبیان فی علوم القرآن", "بیضاوی", "آئینہ قادیانیت", "ہدایہ جلد ثالث"]
ONLY_BOOKS = CONFIG["repair_books"]
WHISPER_MODEL = CONFIG.get("whisper_model", "large-v3")
MAX_HOURS = float(CONFIG.get("max_hours", 3))
REFIX = False
MOVE_GAPS_FOR_REDO = False
WEB_BOOKS = CONFIG["opus_books"]
OPUS_KBPS, MAX_MB = 16, 5.0

print(f"1 · re-transcribe: {len(CONFIG['redo'])} lesson(s)")
for t in CONFIG["redo"]:
    print("     ", t["file"])
print(f"2 · repair: {', '.join(ONLY_BOOKS) or 'all books'}")
print(f"3 · Opus copies: {', '.join(WEB_BOOKS) or 'none'}")

In [ ]:
%%writefile /content/retx_core.py
# Wifaq 1448 · re-transcription functions (shared by the Wifaq Daily notebook and, later, the GitHub pipeline).
# Made from cell_retranscribe.py (5 Oct, used for the الذبائح lessons): same prompt, same checks, same file handling.
# The caller sets OUT_DIR and TX_DIR, calls setup_lanes(get_secret), then redo(name) for each transcript file name.
import os, re, glob, time, shutil, subprocess, datetime, unicodedata
from zoneinfo import ZoneInfo
from google import genai
from google.genai import types

MODELS = ["gemini-3.8-flash", "gemini-3.7-flash", "gemini-3.6-flash"]   # never 3.5 or 2.5 (bad timestamps)
AISTUDIO_SECRET = "GEMINI_API_KEY"
VERTEX_SECRET = "VERTEX_API_KEY"
CHUNK_MIN = 10
WORK = "/content/retx" if os.path.isdir("/content") else "/tmp/retx"
PK = ZoneInfo("Asia/Karachi")
TS_START = re.compile(r"^\s*\[(?:(\d{1,2}):)?(\d{1,2}):(\d{2})\]")
TS_ANY = re.compile(r"\[(?:(\d{1,2}):)?(\d{1,2}):(\d{2})\]")

PROMPT = """This is part {part} of {parts} of a recorded dars in a Pakistani madrasa. The teacher explains in Urdu and reads passages of the Arabic book "{book}" aloud.

Transcribe this clip verbatim.
- Urdu in Urdu script, Arabic in Arabic script, exactly as spoken. Do not translate, summarise or correct anything.
- Every line starts with a timestamp [mm:ss], measured from the start of THIS clip (which is {clip} long).
- Start a new line at every sentence boundary, every time the teacher starts or stops reading Arabic, and at least every 20 seconds. Keep lines short (one or two sentences).
- Put ع: right after the timestamp on every line where the teacher reads the book's Arabic text aloud.
- Include students' questions and the teacher's answers as they are spoken.
- If a word is unclear, write [؟] instead of guessing.
- If the teacher mentions a page number, keep it exactly.
- Output only the transcript lines, nothing else."""


def clock():
    return datetime.datetime.now(PK).strftime("%I:%M %p").lstrip("0").lower()


def say(*a):
    print(clock(), "·", *a, flush=True)


def hms(sec):
    sec = int(round(sec))
    return f"{sec // 3600}:{(sec % 3600) // 60:02d}:{sec % 60:02d}"


def mmss(sec):
    sec = int(round(sec))
    return f"{sec // 60}:{sec % 60:02d}"


def seconds(m):
    return int(m.group(1) or 0) * 3600 + int(m.group(2)) * 60 + int(m.group(3))


def probe(path):
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", path],
                       capture_output=True, text=True)
    return float(r.stdout.strip() or 0)


def nfc(s):
    return unicodedata.normalize("NFC", s)


def find_transcript(name):
    for book_dir in sorted(glob.glob(os.path.join(TX_DIR, "*"))):
        if os.path.basename(book_dir).startswith("_") or not os.path.isdir(book_dir):
            continue
        for p in glob.glob(os.path.join(book_dir, "*.md")):
            if nfc(os.path.basename(p)) == nfc(name):
                return p
    return None


def split_header(text):
    lines = text.split("\n")
    for i, l in enumerate(lines[:80]):
        if l.strip() == "---":
            return lines[:i], lines[i + 1:]
    return [], lines


def field(header_lines, key):
    for l in header_lines:
        l = l.replace("\\", "").strip()
        if l.startswith("- ") and ":" in l and l[2:].split(":", 1)[0].strip() == key:
            return l.split(":", 1)[1].strip()
    return ""


def split_audio(src):
    shutil.rmtree(WORK, ignore_errors=True)
    os.makedirs(WORK)
    local = os.path.join(WORK, "src" + os.path.splitext(src)[1])
    shutil.copyfile(src, local)                       # one read from Drive, then ffmpeg works locally
    subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-i", local, "-vn",
                    "-ac", "1", "-ar", "16000", "-c:a", "libmp3lame", "-b:a", "32k",
                    "-f", "segment", "-segment_time", str(CHUNK_MIN * 60), "-reset_timestamps", "1",
                    os.path.join(WORK, "chunk_%03d.mp3")], check=True)
    total = probe(local)
    os.remove(local)
    return sorted(glob.glob(os.path.join(WORK, "chunk_*.mp3"))), total


def judge(text, clip_sec):
    """-> (ok, n_lines, last_sec, why). Dense, inside the clip, reaching its end, mostly in order."""
    stamps = [seconds(m) for m in (TS_START.match(l) for l in text.split("\n")) if m]
    if not stamps:
        return False, 0, 0, "no line timestamps"
    n, last = len(stamps), max(stamps)
    back = sum(1 for a, b in zip(stamps, stamps[1:]) if b < a - 2)
    need = max(12, int(clip_sec / 40))
    if n < need:
        return False, n, last, f"only {n} lines (want {need}+)"
    if last > clip_sec + 15:
        return False, n, last, f"times run past the clip ({mmss(last)} > {mmss(clip_sec)})"
    if last < 0.8 * clip_sec - 20:
        return False, n, last, f"stops early ({mmss(last)} of {mmss(clip_sec)})"
    if back > 0.1 * n:
        return False, n, last, f"{back} times go backwards"
    return True, n, last, "ok"


# ---- clients: (label, client) for each lane that has a key; built by setup_lanes()
LANES, COMBOS, DEAD = [], [], set()


def setup_lanes(get_secret):
    LANES.clear(); COMBOS.clear(); DEAD.clear()
    k = get_secret(AISTUDIO_SECRET)
    if k:
        LANES.append(("", genai.Client(api_key=k)))
    k = get_secret(VERTEX_SECRET)
    if k:
        LANES.append(("vx:", genai.Client(vertexai=True, api_key=k)))
    COMBOS.extend((lab, cl, m) for m in MODELS for lab, cl in LANES)   # model first, then lane
    return bool(LANES)


def transcribe_part(path, part, parts, book, clip_sec):
    data = open(path, "rb").read()
    prompt = PROMPT.format(part=part, parts=parts, book=book, clip=mmss(clip_sec))
    best = None                                        # (n_lines, text, who, why) of the best rejected answer
    for lab, cl, model in COMBOS:
        who = lab + model
        if who in DEAD:
            continue
        for attempt in range(3):
            try:
                resp = cl.models.generate_content(
                    model=model,
                    contents=[prompt, types.Part.from_bytes(data=data, mime_type="audio/mpeg")],
                    config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=65536))
                text = (resp.text or "").strip()
                break
            except Exception as e:
                msg = str(e)
                if "429" in msg[:60] or "RESOURCE_EXHAUSTED" in msg:
                    say(f"   {who}: out of quota for today, trying the next one")
                    DEAD.add(who); text = None; break
                if any(c in msg[:60] for c in ("400", "401", "403", "404")) or "NOT_FOUND" in msg:
                    say(f"   {who}: not available ({msg[:100]}), trying the next one")
                    DEAD.add(who); text = None; break
                wait = 30 * (attempt + 1)
                say(f"   {who}: {msg[:100]} — retrying in {wait}s")
                time.sleep(wait); text = None
        if who in DEAD or text is None:
            continue
        ok, n, last, why = judge(text, clip_sec)
        if ok:
            say(f"   part {part}/{parts}: {who}, {n} lines, last {mmss(last)} of {mmss(clip_sec)}")
            return text, who, f"part {part}: {n} lines"
        say(f"   part {part}/{parts}: {who} rejected ({why}), trying the next one")
        if best is None or n > best[0]:
            best = (n, text, who, why)
    if best:
        say(f"   part {part}/{parts}: nothing passed; keeping the best answer from {best[2]} ({best[3]})")
        return best[1], best[2], f"part {part}: weak ({best[3]})"
    return None, None, f"part {part}: no answer"


def redo(name):
    say(f"▶ {name}")
    path = find_transcript(name)
    if not path:
        say(f"  ✗ transcript not found under {TX_DIR}/<book>/ — check the name in TARGETS"); return False
    book = os.path.basename(os.path.dirname(path))
    old = open(path, encoding="utf-8").read()
    head, _ = split_header(old)
    src = field(head, "راستہ")
    if not src or not os.path.exists(src):
        say(f"  ✗ recording not found: {src or '(no راستہ line)'} — is the سابعہ shortcut in My Drive?"); return False
    chunks, total = split_audio(src)
    say(f"  {len(chunks)} parts of up to {CHUNK_MIN} min, {mmss(total)} in all")
    texts, used, checks = [], [], []
    for i, ch in enumerate(chunks):
        clip = probe(ch)
        text, who, note = transcribe_part(ch, i + 1, len(chunks), book, clip)
        if text is None:
            say(f"  ✗ stopped: part {i + 1} got no answer from any model (quota?). The old transcript is untouched.")
            return False
        off = i * CHUNK_MIN * 60
        texts.append(TS_ANY.sub(lambda m: "[" + hms(off + seconds(m)) + "]", text))
        used.append(who); checks.append(note)
    body = "\n".join(texts)
    last = max((seconds(m) for m in (TS_START.match(l) for l in body.split("\n")) if m), default=0)
    title = next((l for l in head if l.startswith("#")), f"# {os.path.splitext(name)[0]}")
    keep = [(k, field(head, k)) for k in ("کتاب", "سبق", "تاریخ", "موضوع", "دورانیہ")]
    new_head = [title.replace("\\", ""), ""] + [f"- {k}: {v or '—'}" for k, v in keep] + [
        f"- ماڈل: {', '.join(dict.fromkeys(used))}",
        f"- حصے: {len(chunks)}",
        f"- بنایا: {datetime.date.today().isoformat()} (دوبارہ)",
        f"- راستہ: {src}",
        f"- جانچ: last timestamp {mmss(last)} of {mmss(total)} · " + " · ".join(checks),
    ]
    out = "\n".join(new_head) + "\n\n---\n\n" + body + "\n"
    stamp = datetime.datetime.now(PK).strftime("%Y%m%d-%H%M")
    rep_dir = os.path.join(TX_DIR, "_replaced", book)
    os.makedirs(rep_dir, exist_ok=True)
    shutil.copyfile(path, os.path.join(rep_dir, f"{stamp} {name}"))
    orig = os.path.join(TX_DIR, "_original", book, name)
    if os.path.exists(orig):                               # so _original will hold the new text before repair
        shutil.move(orig, os.path.join(rep_dir, f"{stamp} (original) {name}"))
    tmp = path + ".part"
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(out)
    os.replace(tmp, path)
    say(f"  ✓ saved ({len(body.splitlines())} lines, models: {', '.join(dict.fromkeys(used))}); old copy in _replaced/{book}/")
    return True


def already_redone(name, since):
    """True if the transcript was re-made on or after the date `since` (YYYY-MM-DD)."""
    path = find_transcript(name)
    if not path:
        return False
    head, _ = split_header(open(path, encoding="utf-8").read(6000))
    made = field(head, "بنایا")
    return "دوبارہ" in made and made[:10] >= since

In [ ]:
# @title 4 · Functions (no need to edit)
# ------------------------------------------------------------------ timestamp repair: pure functions
# Gemini writes good text but its timestamps can drift on long clips. Whisper's text is weaker, but its
# timings come from the audio itself. We line Gemini's sentences up against Whisper's words and take
# the time from Whisper. No Gemini request is used.
import re, unicodedata, bisect
import numpy as np
from rapidfuzz import fuzz

# "[0:01:03] text", also "01:03 [0:01:03] text", "- [0:01:03] text", "**[0:01:03]** text" (some models add these)
TS_LINE = re.compile(r"^(\s*(?:[-*•]\s+)?)(?:\*\*)?(?:\(?\d{1,2}:\d{2}(?::\d{2})?\)?\s+)?"
                     r"\[(?:(\d{1,2}):)?(\d{1,4}):(\d{2})\](?:\*\*)?(.*)$")
_FOLD = str.maketrans({
    "ي": "ی", "ى": "ی", "ې": "ی", "ے": "ی", "ئ": "ی", "ك": "ک", "ه": "ہ", "ة": "ہ", "ۃ": "ہ", "ۀ": "ہ",
    "ۂ": "ہ", "ھ": "ہ", "أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا", "ؤ": "و", "ء": None, "ـ": None,
    "٠": None, "١": None, "٢": None, "٣": None, "٤": None, "٥": None, "٦": None, "٧": None, "٨": None, "٩": None,
    "۰": None, "۱": None, "۲": None, "۳": None, "۴": None, "۵": None, "۶": None, "۷": None, "۸": None, "۹": None,
})
_KEEP = re.compile(r"[^\w\s]|\d|_")


def norm_text(s):
    """Fold Arabic/Urdu spelling variants, drop diacritics, punctuation and digits."""
    s = unicodedata.normalize("NFC", s or "")
    s = s.replace("[؟]", " ").replace("ع:", " ")
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn").translate(_FOLD)
    s = _KEEP.sub(" ", s.lower())
    return re.sub(r"\s+", " ", s).strip()


def fmt_hms(sec):
    sec = int(round(max(sec, 0)))
    return f"{sec // 3600}:{(sec % 3600) // 60:02d}:{sec % 60:02d}"


def split_transcript(text):
    """-> (header, body_lines). Header is everything up to and including the '---' line."""
    lines = text.split("\n")
    for i, l in enumerate(lines[:80]):
        if l.strip() == "---":
            return "\n".join(lines[:i + 1]), lines[i + 1:]
    return "", lines


def parse_lines(body_lines):
    """Timestamped lines -> list of dicts with index into body_lines, old seconds, normalized text."""
    out = []
    for k, line in enumerate(body_lines):
        m = TS_LINE.match(line)
        if not m:
            continue
        sec = int(m.group(2) or 0) * 3600 + int(m.group(3)) * 60 + int(m.group(4))
        out.append({"k": k, "old": sec, "raw": m.group(5), "norm": norm_text(m.group(5))})
    return out


# ------------------------------------------------------------------ whisper side
class WhisperText:
    """Whisper words joined into one normalized string, with a time for every character."""

    def __init__(self, words):
        parts, times = [], []
        for start, end, w in words:
            t = norm_text(w)
            if not t:
                continue
            if parts:
                parts.append(" ")
                times.append(start)
            span = max(end - start, 0.01)
            for j, _ in enumerate(t):
                times.append(start + span * j / max(len(t), 1))
            parts.append(t)
        self.text = "".join(parts)
        self.times = np.array(times, dtype=float) if times else np.zeros(1)

    def time_at(self, i):
        if len(self.text) == 0:
            return 0.0
        i = int(min(max(round(i), 0), len(self.text) - 1))
        return float(self.times[i])

    def char_at_time(self, t):
        return int(np.searchsorted(self.times, t))


# ------------------------------------------------------------------ alignment
def _weighted_lis(anchors):
    """Keep the heaviest subset of anchors whose Whisper positions rise with the line order."""
    n = len(anchors)
    if n == 0:
        return []
    best = [a["score"] for a in anchors]
    prev = [-1] * n
    for i in range(n):
        for j in range(max(0, i - 60), i):           # neighbours are enough; keeps it fast
            if anchors[j]["w0"] < anchors[i]["w0"] and best[j] + anchors[i]["score"] > best[i]:
                best[i], prev[i] = best[j] + anchors[i]["score"], j
    i = int(np.argmax(best))
    keep = []
    while i != -1:
        keep.append(anchors[i])
        i = prev[i]
    return keep[::-1]


def align(lines, wt, duration, min_chars=10):
    """Find each Gemini line in Whisper's text. Returns (new_times, anchors, stats)."""
    W, n = wt.text, len(lines)
    if not lines or len(W) < 50:
        return None, [], {"reason": "Whisper heard almost nothing"}
    # Gemini character positions (cumulative), so a line between two anchors can be placed in proportion.
    cg, pos = [], 0
    for ln in lines:
        cg.append(pos)
        pos += len(ln["norm"]) + 1
    g_total = max(pos, 1)
    ratio = len(W) / g_total                          # Whisper chars per Gemini char, roughly 1
    def find(q, lo, hi, need):
        lo, hi = int(max(0, lo)), int(min(len(W), hi))
        if hi - lo < len(q):
            return None
        r = fuzz.partial_ratio_alignment(q, W[lo:hi], score_cutoff=min(need, 100))
        return None if r is None else (r.score, lo + r.dest_start, lo + r.dest_end)

    def need_for(q):
        return 64 if len(q) >= 45 else (72 if len(q) >= 25 else 82)

    def confirmed(i, w_end, g_end):
        """After a long jump, the next lines must also be found right after it."""
        hits = tries = 0
        for j in range(i + 1, n):
            q2 = lines[j]["norm"]
            if len(q2.replace(" ", "")) < 15:
                continue
            tries += 1
            e2 = w_end + (cg[j] - g_end) * ratio
            if find(q2, e2 - 1200, e2 + 1200 + len(q2), need_for(q2)):
                hits += 1
            if tries == 4:
                break
        return tries > 0 and hits >= min(2, tries)

    anchors = []
    last_w, last_g = 0, 0
    for i, ln in enumerate(lines):
        q = ln["norm"]
        if len(q.replace(" ", "")) < min_chars:
            continue
        need = need_for(q)
        expect = last_w + (cg[i] - last_g) * ratio
        half = max(1500, 0.6 * (cg[i] - last_g) * ratio, len(q) * 4)
        hit = find(q, min(expect - half, len(W) - len(q)), expect + half + len(q), need)
        if hit is None and len(q) >= 25:
            # Gemini may have skipped a stretch: look further ahead, but ask for a closer match
            # and check that the following lines carry on from there.
            far = find(q, expect + half, expect + 9000 + len(q), need + 6)
            if far and confirmed(i, far[2], cg[i] + len(q)):
                hit = far
        if hit is None:
            continue
        score, w0, w1 = hit
        anchors.append({"i": i, "w0": w0, "w1": w1, "score": score * min(len(q), 120) / 120})
        last_w, last_g = w1, cg[i] + len(q)
    eligible = sum(1 for ln in lines if len(ln["norm"].replace(" ", "")) >= min_chars)
    kept = _weighted_lis(anchors)
    stats = {"eligible": eligible, "found": len(anchors), "kept": len(kept),
             "coverage": round(len(kept) / max(eligible, 1), 3)}
    if len(kept) < max(8, 0.25 * eligible):
        stats["reason"] = f"only {len(kept)} of {eligible} lines could be matched"
        return None, kept, stats
    # place every line: anchors exactly, the rest in proportion between neighbouring anchors
    ka = [(a["i"], a["w0"]) for a in kept]
    idx = [a[0] for a in ka]
    new = []
    for i in range(n):
        j = bisect.bisect_left(idx, i)
        if j < len(idx) and idx[j] == i:
            w = ka[j][1]
        elif j == 0:                                   # before the first anchor
            i1, w1 = ka[0]
            w = w1 - (cg[i1] - cg[i]) * ratio
        elif j == len(idx):                            # after the last anchor
            i0, w0 = ka[-1]
            w = w0 + (cg[i] - cg[i0]) * ratio
        else:
            (i0, w0), (i1, w1) = ka[j - 1], ka[j]
            f = (cg[i] - cg[i0]) / max(cg[i1] - cg[i0], 1)
            w = w0 + f * (w1 - w0)
        new.append(min(max(wt.time_at(w), 0.0), duration))
    for i in range(1, n):                              # never go backwards
        new[i] = max(new[i], new[i - 1])
    shifts = [abs(new[i] - lines[i]["old"]) for i in range(n)]
    stats.update({"median_shift": round(float(np.median(shifts)), 1), "max_shift": round(float(max(shifts)), 1),
                  "end_old": lines[-1]["old"], "end_new": round(new[-1], 1)})
    return new, kept, stats


def find_gaps(lines, kept, wt, duration, min_sec=75):
    """Stretches where Whisper heard a lot of speech but Gemini wrote little: likely skipped text."""
    if not kept:
        return []
    cg, pos = [], 0
    for ln in lines:
        cg.append(pos)
        pos += len(ln["norm"]) + 1
    g_end = pos
    ratio = len(wt.text) / max(g_end, 1)
    pts = [(-1, 0, 0)] + [(a["i"], a["w0"], cg[a["i"]]) for a in kept] + [(len(lines), len(wt.text), g_end)]
    gaps = []
    for (ia, wa, ga), (ib, wb, gb) in zip(pts, pts[1:]):
        ta, tb = wt.time_at(wa), (wt.time_at(wb) if wb < len(wt.text) else duration)
        w_chars, g_chars = wb - wa, (gb - ga) * ratio
        if tb - ta >= min_sec and w_chars >= 700 and g_chars < 0.35 * w_chars:
            gaps.append((round(ta), round(tb)))
    return gaps


def render(header, body_lines, lines, new_times, note_lines):
    body = list(body_lines)
    for ln, t in zip(lines, new_times):
        m = TS_LINE.match(body[ln["k"]])
        body[ln["k"]] = f"{m.group(1)}[{fmt_hms(t)}]{m.group(5)}"
    head = header.split("\n")
    # put our notes just above the closing '---' (after an existing '- جانچ:' line, if any)
    while head and head[-1].strip() in ("", "---"):
        head.pop()
    head = [h for h in head if not h.startswith("- وقت:") and not h.startswith("- خلا:")]
    return "\n".join(head + note_lines + ["", "---"] + body)


# ------------------------------------------------------------------ timestamp repair: the run
import os, csv, json, glob, time, shutil, hashlib, subprocess, datetime
from zoneinfo import ZoneInfo

PK = ZoneInfo("Asia/Karachi")
TX_DIR = os.path.join(OUT_DIR, "transcripts")
ORIG_DIR = os.path.join(TX_DIR, "_original")     # untouched copies of every transcript we change
REDO_DIR = os.path.join(TX_DIR, "_redo")         # transcripts moved here are done again by the transcription notebook
WH_DIR = os.path.join(OUT_DIR, "_whisper")       # Whisper's word timings, kept so nothing is heard twice
LOG_PATH = os.path.join(OUT_DIR, "timestamp_fix.csv")
REPORT_PATH = os.path.join(OUT_DIR, "timestamp_fix.md")
LOCAL = "/content/tsfix" if os.path.isdir("/content") else "/tmp/tsfix"
BROKEN_MARK = "ٹرانسکرپٹ نہیں بن سکا"
HARD_FAILS = ("no timestamps", "empty answer", "blocked")


def now_pk():
    return datetime.datetime.now(PK)


def clock(dt):
    return dt.strftime("%I:%M %p").lstrip("0").lower()


def _norm_name(s):
    return norm_text(s).replace(" ", "")


def book_rank(book):
    nb = _norm_name(book)
    for i, b in enumerate(BOOK_ORDER):
        if _norm_name(b) and _norm_name(b) in nb:
            return i
    return len(BOOK_ORDER)


def header_fields(header):
    f = {}
    for line in header.split("\n"):
        if line.startswith("- ") and ":" in line:
            k, v = line[2:].split(":", 1)
            f[k.strip()] = v.strip()
    return f


def list_transcripts():
    rows = []
    for book_dir in sorted(glob.glob(os.path.join(TX_DIR, "*"))):
        book = os.path.basename(book_dir)
        if not os.path.isdir(book_dir) or book.startswith("_"):
            continue
        if ONLY_BOOKS and not any(_norm_name(b) in _norm_name(book) for b in ONLY_BOOKS):
            continue
        for path in glob.glob(os.path.join(book_dir, "*.md")):
            try:
                with open(path, encoding="utf-8") as f:
                    head = f.read(4000)
            except OSError:
                continue
            fields = header_fields(split_transcript(head)[0] or head)
            lesson = fields.get("سبق", "")
            rows.append({"path": path, "book": book, "file": os.path.basename(path),
                         "lesson": int(lesson) if lesson.isdigit() else 9999,
                         "audio": fields.get("راستہ", ""), "model": fields.get("ماڈل", ""),
                         "check": fields.get("جانچ", ""), "fixed": "وقت" in fields})
    rows.sort(key=lambda r: (book_rank(r["book"]), r["lesson"], r["file"]))
    return rows


def unfinished(text, check):
    """True when the transcription notebook will redo this transcript anyway."""
    if BROKEN_MARK in text or any(h in check for h in HARD_FAILS):
        return True
    for a, b in re.findall(r"(?<!nothing transcribed \()stopped at (\d+:\d\d) of (\d+:\d\d)", check):
        sec = lambda x: int(x.split(":")[0]) * 60 + int(x.split(":")[1])
        if sec(a) < 0.25 * sec(b):
            return True
    return False


# ------------------------------------------------------------------ Whisper
_PIPE = []


def _preload_cuda_libs():
    """Colab sometimes can't find cuDNN for faster-whisper; load the copies pip installed with torch."""
    import ctypes, site
    for sp in site.getsitepackages():
        for pat in ("nvidia/cublas/lib/libcublas*.so*", "nvidia/cudnn/lib/libcudnn*.so*"):
            for lib in sorted(glob.glob(os.path.join(sp, pat))):
                try:
                    ctypes.CDLL(lib, mode=ctypes.RTLD_GLOBAL)
                except OSError:
                    pass


def whisper_pipe():
    if _PIPE:
        return _PIPE[0]
    _preload_cuda_libs()
    import torch
    from faster_whisper import WhisperModel, BatchedInferencePipeline
    if torch.cuda.is_available():
        name, dev, ctype, batch = WHISPER_MODEL, "cuda", "float16", 16
    else:
        print("⚠ No GPU in this runtime (Runtime → Change runtime type → T4 GPU). "
              "Using the small model on the CPU — this is slow and less accurate.")
        name, dev, ctype, batch = "small", "cpu", "int8", 4
    print(f"Loading Whisper {name} on {dev}…", flush=True)
    model = WhisperModel(name, device=dev, compute_type=ctype)
    _PIPE.append((BatchedInferencePipeline(model=model), batch, name))
    return _PIPE[0]


def probe_seconds(path):
    try:
        out = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                              "-of", "default=nw=1:nk=1", path], capture_output=True, text=True, timeout=300).stdout
        return float(out.strip())
    except Exception:
        return 0.0


def local_copy(src):
    """Copy the recording to the Colab disk first: reading straight from Drive can cut audio short."""
    os.makedirs(LOCAL, exist_ok=True)
    dst = os.path.join(LOCAL, "src_" + hashlib.md5(src.encode("utf-8")).hexdigest()[:12] + os.path.splitext(src)[1])
    want = os.path.getsize(src)
    for _ in range(3):
        if os.path.exists(dst) and os.path.getsize(dst) == want:
            return dst
        try:
            shutil.copyfile(src, dst + ".part")
            os.replace(dst + ".part", dst)
        except OSError as e:
            print(f"   … copying from Drive failed ({e}); trying again", flush=True)
            time.sleep(10)
    if os.path.exists(dst) and os.path.getsize(dst) == want:
        return dst
    raise RuntimeError("the recording could not be copied from Drive completely")


def load_audio(path, sr=16000):
    """Decode with ffmpeg (not PyAV: Colab's newer PyAV breaks faster-whisper's own decoder)."""
    out = subprocess.run(["ffmpeg", "-nostdin", "-v", "error", "-i", path, "-vn", "-ac", "1", "-ar", str(sr),
                          "-f", "f32le", "-"], capture_output=True, check=True, timeout=3600).stdout
    return np.frombuffer(out, np.float32).copy()


def run_whisper(audio_path):
    """-> (duration_sec, [(start, end, word), ...], model_name)"""
    pipe, batch, name = whisper_pipe()
    audio = load_audio(audio_path)
    segs, info = pipe.transcribe(audio, language="ur", batch_size=batch, word_timestamps=True)
    words = []
    for s in segs:
        if getattr(s, "words", None):
            words += [(round(w.start, 2), round(w.end, 2), w.word) for w in s.words]
        else:
            words.append((round(s.start, 2), round(s.end, 2), s.text))
    return len(audio) / 16000, words, name


def whisper_for(row):
    cache = os.path.join(WH_DIR, row["book"], os.path.splitext(row["file"])[0] + ".json")
    if os.path.exists(cache):
        with open(cache, encoding="utf-8") as f:
            d = json.load(f)
        return d["duration"], [tuple(w) for w in d["words"]], d.get("model", ""), True
    src = row["audio"]
    if not src or not os.path.exists(src):
        raise FileNotFoundError(f"recording not found: {src or '(no path in the header)'}")
    local = local_copy(src)
    try:
        t0 = time.time()
        dur, words, name = run_whisper(local)
        dur = dur or probe_seconds(local)
        print(f"   Whisper heard {len(words)} words in {time.time() - t0:.0f}s", flush=True)
    finally:
        try:
            os.remove(local)
        except OSError:
            pass
    os.makedirs(os.path.dirname(cache), exist_ok=True)
    tmp = cache + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump({"file": row["file"], "audio": src, "duration": dur, "model": name, "words": words},
                  f, ensure_ascii=False)
    os.replace(tmp, cache)
    return dur, words, name, False


# ------------------------------------------------------------------ one transcript
def _log(row, status, stats=None, gaps=(), note=""):
    stats = stats or {}
    new = not os.path.exists(LOG_PATH)
    with open(LOG_PATH, "a", newline="", encoding="utf-8-sig" if new else "utf-8") as f:
        w = csv.writer(f)
        if new:
            w.writerow(["time_pk", "book", "lesson", "file", "transcribed_by", "lines", "matched_pct",
                        "typical_shift_s", "biggest_shift_s", "end_before", "end_after", "gaps", "status", "note"])
        w.writerow([now_pk().strftime("%Y-%m-%d %I:%M %p"), row["book"], row["lesson"] if row["lesson"] != 9999 else "",
                    row["file"], row["model"], stats.get("eligible", ""),
                    round(100 * stats["coverage"]) if "coverage" in stats else "",
                    stats.get("median_shift", ""), stats.get("max_shift", ""),
                    fmt_hms(stats["end_old"]) if "end_old" in stats else "",
                    fmt_hms(stats["end_new"]) if "end_new" in stats else "",
                    " ".join(f"{fmt_hms(a)}–{fmt_hms(b)}" for a, b in gaps), status, note])


def fix_one(row):
    with open(row["path"], encoding="utf-8") as f:
        text = f.read()
    stamp = (os.path.getmtime(row["path"]), len(text))
    if unfinished(text, row["check"]):
        return "skipped", "the transcription notebook will redo it"
    header, body = split_transcript(text)
    lines = parse_lines(body)
    if len(lines) < 15:
        return "skipped", f"only {len(lines)} timestamped lines found"
    dur, words, wmodel, cached = whisper_for(row)
    wt = WhisperText(words)
    new, kept, stats = align(lines, wt, dur)
    if new is None:
        _log(row, "not aligned", stats, note=stats.get("reason", ""))
        return "not aligned", stats.get("reason", "")
    gaps = find_gaps(lines, kept, wt, dur)
    notes = [f"- وقت: Whisper ({wmodel}) سے ملایا، {now_pk().date().isoformat()} · {round(100 * stats['coverage'])}% سطریں ملیں"
             f" · سب سے بڑی تبدیلی {fmt_hms(stats['max_shift'])} · اصل: transcripts/_original"]
    if gaps:
        notes.append("- خلا: " + " · ".join(f"{fmt_hms(a)}–{fmt_hms(b)}" for a, b in gaps)
                     + " (یہاں آواز ہے مگر متن بہت کم — دوبارہ ٹرانسکرائب کرنا بہتر)")
    out = render(header, body, lines, new, notes)
    # the transcription notebook may have rewritten the file meanwhile: then leave it for the next run
    with open(row["path"], encoding="utf-8") as f:
        again = f.read()
    if (os.path.getmtime(row["path"]), len(again)) != stamp:
        return "skipped", "the transcript changed while we worked; next run"
    orig = os.path.join(ORIG_DIR, row["book"], row["file"])
    if not os.path.exists(orig):
        os.makedirs(os.path.dirname(orig), exist_ok=True)
        shutil.copyfile(row["path"], orig)
    tmp = row["path"] + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(out)
    os.replace(tmp, row["path"])
    status = "fixed, text missing" if gaps else "fixed"
    if gaps and MOVE_GAPS_FOR_REDO:
        dst = os.path.join(REDO_DIR, row["book"], row["file"])
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.move(row["path"], dst)
        status += ", moved for redo"
    _log(row, status, stats, gaps)
    return status, (f"{round(100 * stats['coverage'])}% matched · typical shift {stats['median_shift']:.0f}s · "
                    f"biggest {fmt_hms(stats['max_shift'])}" + (f" · text missing at " + ", ".join(
                        f"{fmt_hms(a)}–{fmt_hms(b)}" for a, b in gaps) if gaps else ""))


# ------------------------------------------------------------------ report
def write_report():
    if not os.path.exists(LOG_PATH):
        return
    with open(LOG_PATH, encoding="utf-8-sig") as f:
        rows = list(csv.DictReader(f))
    last = {}
    for r in rows:                                     # latest result per transcript
        last[(r["book"], r["file"])] = r
    out = [f"# Timestamp repair — {now_pk().strftime('%d %b %Y, ')}{clock(now_pk())}", ""]
    fixed = [r for r in last.values() if r["status"].startswith("fixed")]
    gappy = [r for r in fixed if r["gaps"]]
    bad = [r for r in last.values() if r["status"] == "not aligned"]
    out += [f"- Fixed: {len(fixed)} transcripts ({len(gappy)} of them have stretches with text missing)",
            f"- Could not be matched: {len(bad)}", ""]
    if gappy:
        out += ["## Text missing (worth transcribing again)", "",
                "| کتاب | سبق | فائل | کہاں | ماڈل |", "|---|---|---|---|---|"]
        out += [f"| {r['book']} | {r['lesson']} | {r['file']} | {r['gaps']} | {r['transcribed_by']} |" for r in gappy]
        out.append("")
    if bad:
        out += ["## Could not be matched", ""] + [f"- {r['book']} · {r['file']}: {r['note']}" for r in bad] + [""]
    big = sorted(fixed, key=lambda r: -float(r["biggest_shift_s"] or 0))[:25]
    if big:
        out += ["## Biggest corrections", "", "| کتاب | فائل | ماڈل | سب سے بڑی تبدیلی | آخری وقت پہلے → اب |", "|---|---|---|---|---|"]
        out += [f"| {r['book']} | {r['file']} | {r['transcribed_by']} | {fmt_hms(float(r['biggest_shift_s'] or 0))} | "
                f"{r['end_before']} → {r['end_after']} |" for r in big]
    tmp = REPORT_PATH + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        f.write("\n".join(out) + "\n")
    os.replace(tmp, REPORT_PATH)


# ------------------------------------------------------------------ main loop
def run_fix():
    deadline = time.time() + MAX_HOURS * 3600
    rows = list_transcripts()
    todo = [r for r in rows if REFIX or not r["fixed"]]
    print(f"{len(rows)} transcripts in Drive, {len(todo)} to check"
          + (f" (books: {', '.join(ONLY_BOOKS)})" if ONLY_BOOKS else "") + ".")
    print(f"Stops by itself after {MAX_HOURS} h (at {clock(now_pk() + datetime.timedelta(hours=MAX_HOURS))}); "
          "stopping earlier is safe.\n", flush=True)
    done = {}
    try:
        for n, row in enumerate(todo, 1):
            if time.time() > deadline:
                print(f"\nTime limit reached (MAX_HOURS = {MAX_HOURS}). Run it again to carry on.")
                break
            print(f"[{n}/{len(todo)}] {row['book']} · {row['file']}", flush=True)
            try:
                status, note = fix_one(row)
            except Exception as e:
                status, note = "error", f"{type(e).__name__}: {str(e)[:160]}"
                _log(row, "error", note=note)
            done[status] = done.get(status, 0) + 1
            print(f"   {status}" + (f" — {note}" if note else ""), flush=True)
    except KeyboardInterrupt:
        print("\n■ Stopped. Finished transcripts are saved; run again to carry on.")
    write_report()
    print("\n" + ", ".join(f"{k}: {v}" for k, v in done.items()))
    print(f"Report: {REPORT_PATH}")


import csv, glob, json, math, os, shutil, subprocess
OPUS_DIR = os.path.join(OUT_DIR, "web_audio_opus")


def _opus_targets():
    with open(os.path.join(OUT_DIR, "recordings_inventory.csv"), encoding="utf-8-sig") as f:
        recs = list(csv.reader(f))[1:]
    out = []
    for rec in recs:
        rec = rec + [""] * (12 - len(rec))
        file, book, date, path = rec[0], rec[1], rec[10], rec[11]
        ranks = [i for i, b in enumerate(WEB_BOOKS) if _norm_name(b) and _norm_name(b) in _norm_name(book)]
        if path and ranks:
            out.append((ranks[0], date or "9999", file, book, path))
    return sorted(out)


def _encode_opus(src, start, seconds, dst, kbps):
    tmp = os.path.join(LOCAL, "opus_tmp.ogg")
    subprocess.run(["ffmpeg", "-nostdin", "-v", "error", "-y", "-ss", f"{start:.2f}", "-t", f"{seconds:.2f}", "-i", src,
                    "-vn", "-ac", "1", "-ar", "48000", "-c:a", "libopus", "-b:a", f"{kbps}k",
                    "-application", "voip", "-frame_duration", "60", tmp],
                   check=True, capture_output=True, timeout=3600)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copyfile(tmp, dst + ".part")
    os.replace(dst + ".part", dst)
    return os.path.getsize(dst)


def make_opus_audio():
    os.makedirs(LOCAL, exist_ok=True)
    targets = _opus_targets()
    print(f"{len(targets)} recordings in {', '.join(WEB_BOOKS)}", flush=True)
    made = 0
    for n, (_, _, file, book, path) in enumerate(targets, 1):
        stem = os.path.splitext(file)[0]
        side = os.path.join(OPUS_DIR, book, stem + ".json")
        if os.path.exists(side):
            continue
        local = None
        try:
            local = local_copy(path)
            dur = probe_seconds(local)
            parts = max(1, math.ceil((dur * OPUS_KBPS * 125 * 1.04 + 30000) / (MAX_MB * 1e6)))
            step = dur / parts
            info = []
            for i in range(parts):
                name = f"{stem}.ogg" if parts == 1 else f"{stem} - {i + 1}.ogg"
                dst = os.path.join(OPUS_DIR, book, name)
                length = step if i < parts - 1 else dur - i * step + 1
                size = _encode_opus(local, i * step, length, dst, OPUS_KBPS)
                if size > MAX_MB * 1e6:      # rare: squeeze a little more
                    size = _encode_opus(local, i * step, length, dst, 12)
                info.append({"web_file": f"{book}/{name}", "start": round(i * step, 2), "size_mb": round(size / 1e6, 2)})
            with open(side, "w", encoding="utf-8") as f:
                json.dump({"file": file, "book": book, "seconds": round(dur, 1), "parts": info}, f, ensure_ascii=False)
            made += 1
            print(f"[{n}/{len(targets)}] {file}: {len(info)} part(s), {sum(p['size_mb'] for p in info):.1f} MB", flush=True)
        except Exception as e:
            print(f"[{n}/{len(targets)}] {file}: ✗ {type(e).__name__}: {str(e)[:150]}", flush=True)
        finally:
            try:
                if local:
                    os.remove(local)
            except Exception:
                pass
    rows = []
    for side in glob.glob(os.path.join(OPUS_DIR, "*", "*.json")):
        with open(side, encoding="utf-8") as f:
            d = json.load(f)
        for i, p in enumerate(d["parts"], 1):
            rows.append([d["book"], d["file"], i, p["start"], d["seconds"], p["size_mb"], p["web_file"]])
    with open(os.path.join(OPUS_DIR, "index.csv"), "w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f)
        w.writerow(["book", "file", "part", "start_sec", "lesson_seconds", "size_mb", "web_file"])
        w.writerows(sorted(rows))
    print(f"\n✓ {made} new, {len(rows)} files listed in web_audio_opus/index.csv")

print('✓ ready')

In [ ]:
# @title 5 · Re-transcribe (Gemini) — safe to stop and run again
import importlib, sys
sys.path.insert(0, "/content")
import retx_core as rx
importlib.reload(rx)
rx.OUT_DIR, rx.TX_DIR = OUT_DIR, TX_DIR

def _secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

DONE_LOG = []
todo = [t for t in CONFIG["redo"] if not rx.already_redone(t["file"], t.get("added", "2000-01-01"))]
if not todo:
    print("Nothing to re-transcribe today.")
elif not rx.setup_lanes(_secret):
    print("No API key found: add GEMINI_API_KEY (and VERTEX_API_KEY) in Colab Secrets with Notebook access. Skipping step 1.")
else:
    rx.say(f"{len(todo)} to re-transcribe · lanes: {', '.join('Vertex express' if l else 'AI Studio' for l, _ in rx.LANES)}")
    for t in todo:
        ok = rx.redo(t["file"])
        DONE_LOG.append(("✓" if ok else "✗") + " " + t["file"])
        if not ok and len(rx.DEAD) >= len(rx.COMBOS):
            rx.say("All models are out of quota for today; the rest waits for tomorrow.")
            break
    print(f"\n{sum(1 for x in DONE_LOG if x.startswith('✓'))} of {len(todo)} re-transcribed.")

In [ ]:
# @title 6 · Repair timestamps (GPU) — stops by itself after max_hours
run_fix()

In [ ]:
# @title 7 · Opus copies for the card site
make_opus_audio()

In [ ]:
# @title 8 · Note for Claude
import csv
now = datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=5)))
lines = [f"# Daily run {now:%d %b %Y, %I:%M %p}", "", "## Re-transcribed"] + [f"- {x}" for x in DONE_LOG or ["(none)"]]
try:
    with open(os.path.join(OUT_DIR, "timestamp_fix.csv"), encoding="utf-8-sig") as f:
        rows = [r for r in csv.DictReader(f) if r.get("time_pk", "").startswith(now.strftime("%Y-%m-%d"))]
    lines += ["", f"## Repaired today: {len(rows)}"] + [f"- {r['file']}: {r['status']} {r['matched_pct']}%" for r in rows]
except Exception as e:
    lines += ["", f"(could not read timestamp_fix.csv: {e})"]
with open(os.path.join(AUTO_DIR, "daily_done.md"), "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")
print("\n".join(lines))
print("\n✓ Finished. Now run the transcription notebook (v2.5) as usual.")